# Lab09: Action Recognition

This lab introduces the fundamentals of action recognition using video transformers. Instead of real videos, you will work with 3D medical images (OrganMNIST3D), which are treated as short video sequences by interpreting slices as frames. <br> 
While this is not true action recognition (as there is no real motion), it allows you to experiment with spatio-temporal modeling without the computational cost of large video datasets. <br> Using the ViViT (Video Vision Transformer) architecture, the lab focuses on how video data is structured and how transformer-based models can be applied to sequence classification tasks.

In [ ]:
# uncomment the following lines to install the required packages
#%pip install medmnist
#%pip install transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms

from medmnist import OrganMNIST3D

from transformers import VivitConfig, VivitModel

In [ ]:
# Setting seed for reproducibility
SEED = 42
torch.manual_seed(SEED)

# Data

In OrganMNIST3D, each sample is a 3D image of size 28×28×28, which can be interpreted as a video with 28 frames (slices), each of size 28×28 pixels. The dataset contains 11 classes corresponding to different organs. The task is to classify each 3D image into one of the 11 organ classes.

In [ ]:
train_dataset = OrganMNIST3D(split="train", download=True)
val_dataset = OrganMNIST3D(split="val", download=True)

In [ ]:
from medmnist import INFO

# print classes and labels info
info = INFO['organmnist3d']
print(info['label'])

In [ ]:
BATCH_SIZE = 32

# cast to tensor and normalize using mean .5, std .5
data_transform = transforms.Compose([

])

# define train and validation dataloaders
train_loader = 
val_loader = 

# Model

Extend the main ViViT implementation to work on a 11 classes classification task. <br>
Refer to the main documentation of the ViVit model (https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/vivit) to complete the task.

In [ ]:
class ViViTForVideoRecognition(nn.Module):

    def __init__(self, hidden_size = 512, output_size = 11):
        super().__init__()

        # add vivit config
        config_obj = {
            "image_size": 28,
            "num_frames": 28, # 28 slices in the 3D image
            # tubulet size defines how the model chunks the input:
            # 2 frames at a time (temporal depth)
            # 7×7 pixels (spatial size)
            # How many tokens does it create?
            # Time: 28 / 2 = 14
            # Height: 28 / 7 = 4
            # Width: 28 / 7 = 4
            "tubelet_size": (2, 7, 7), # 224 tokens, 16 (4*4) spatial and 14 temporal chunks
            "num_channels": 1,
            "hidden_size": 512,
            "num_hidden_layers": 6,
            "num_attention_heads": 4
        }
        config = 
        
        # define vivit model
        self.vivit = 

        # remove the unnecessary pooler block
        # hint: use an identity function to replace the pooler
        self.vivit.pooler = 

        self.lin = nn.Linear(hidden_size, output_size)
        self.activation = nn.Softmax(-1)

    def forward(self, x):
        x = self.vivit(x)

        # CLS token
        cls = 
        
        x = self.lin(cls)
        x = self.activation(x)

        return x

In [ ]:
vivit = ViViTForVideoRecognition().type(torch.float32)
# print model summary


In [ ]:
# print model parameters


# Train

In [ ]:
def val(epoch, data, model, device = "cuda"):
    model.eval()

    hist_loss = 0
    hist_acc = 0

    for idx, (samples, labels) in enumerate(data):
        
        #- move data to gpu (make sure the shapes are correct)

        # forward
        preds = model(samples)

        #- loss
        loss = 

        # metric
        max_preds = torch.argmax(preds, dim=1)
        acc = (max_preds == labels).sum()/BATCH_SIZE

        # stats
        hist_loss += loss.item()
        hist_acc += acc.item()

    hist_loss /= idx
    hist_acc /= idx

    print(f"[Val {epoch}] loss: {hist_loss}, acc: {hist_acc}")

    return hist_loss, hist_acc

In [ ]:
device = "cuda"
epochs = 150
log_frequency = len(train_loader)/2

# stats
history = {
    "train": {
        "acc": [0 for _ in range(epochs)],
        "loss": [0 for _ in range(epochs)],
    },
    "val": {
        "acc": [0 for _ in range(epochs)],
        "loss": [0 for _ in range(epochs)],
    }
}

opt = optim.Adam(vivit.parameters(), lr=1e-4)

vivit.to(device)

vivit.train()
for epoch in range(epochs):
    for idx, (samples, labels) in enumerate(train_loader):  

        # clean grads
        opt.zero_grad()
        
        #- move data to gpu (make sure the shapes are correct)

        # forward
        preds = vivit(samples)

        #- loss
        loss =

        # backward
        loss.backward()
        opt.step()

        max_preds = torch.argmax(preds, dim=1)
        acc = (max_preds == labels).sum()/BATCH_SIZE
        
        # stats
        history["train"]["loss"][epoch] += loss.item()
        history["train"]["acc"][epoch] += acc.item()

        if idx % log_frequency == 0:
            print(f"[{epoch + 1}/{epochs}] loss:{loss.item():.5f}, acc:{acc.item():.5f}")

    history["train"]["loss"][epoch] /= idx
    history["train"]["acc"][epoch] /= idx

    history["val"]["loss"][epoch], history["val"]["acc"][epoch] = val(epoch + 1, val_loader, vivit)

# Plot

In [12]:
from matplotlib import pyplot as plt

In [ ]:
plt.figure(figsize=(15, 5))
plt.suptitle("ViViT - OrganMNIST3D")

# loss
plt.subplot(1, 2, 1)
plt.title("Loss")
plt.plot(history["train"]["loss"], label="train")
plt.plot(history["val"]["loss"], label="val")
plt.legend()

# accuracy
plt.subplot(1, 2, 2)
plt.title("Accuracy")
plt.plot(history["train"]["acc"], label="train")
plt.plot(history["val"]["acc"], label="val")
plt.legend()

plt.show()